In [ ]:
import requests
import json
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
from typing import List, Dict, Optional

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
}

BASE_URL = "https://www.jumia.ci"

def load_categories(filepath: str) -> Dict[str, str]:
    """
    Charge les catégories à partir d'un fichier JSON.

    Args:
        filepath (str): Chemin vers le fichier JSON.

    Returns:
        dict: Dictionnaire de catégories et leurs URL associées.
    """
    with open(filepath, "r", encoding="utf-8") as f:
        return json.load(f)

def fetch_html(url: str, retries: int = 3, delay: float = 2.0) -> Optional[str]:
    """
    Récupère le contenu HTML d'une URL avec gestion des erreurs et retries.

    Args:
        url (str): L'URL à récupérer.
        retries (int): Nombre de tentatives en cas d'échec.
        delay (float): Délai entre les tentatives.

    Returns:
        str | None: Le contenu HTML ou None si échec.
    """
    for attempt in range(retries):
        try:
            response = requests.get(url, headers=HEADERS, timeout=10)
            response.raise_for_status()
            return response.text
        except requests.RequestException as e:
            print(f"[ERREUR] Tentative {attempt+1}/{retries} échouée pour {url}: {e}")
            time.sleep(delay + random.uniform(0.5, 1.5))
    return None

def parse_products(html: str) -> List[Dict[str, str]]:
    """
    Parse le contenu HTML pour extraire les données des produits.

    Args:
        html (str): Le contenu HTML de la page Jumia.

    Returns:
        list: Liste de dictionnaires représentant les produits.
    """
    soup = BeautifulSoup(html, "lxml")
    products = soup.find_all("article", class_="prd")
    print(f"{len(products)} produits trouvés sur la page.")

    data = []

    for product in products:
        try:
            name_tag = product.select_one("a.core > div.name")
            price_tag = product.select_one("div.prc")
            old_price_tag = product.select_one("div.old")
            rating_tag = product.select_one("div.stars._s")
            link_tag = product.select_one("a.core")

            data.append({
                "name": name_tag.text.strip() if name_tag else "N/A",
                "brand": link_tag.get("data-gtm-brand", "N/A") if link_tag else "N/A",
                "price": price_tag.text.strip() if price_tag else "N/A",
                "old_price": old_price_tag.text.strip() if old_price_tag else "N/A",
                "link": BASE_URL + link_tag["href"] if link_tag and link_tag.get("href") else "N/A",
                "rating": rating_tag.text.strip() if rating_tag else "N/A",
            })
        except Exception as e:
            print(f"[WARNING] Erreur pendant le parsing d’un produit : {e}")

    return data

def save_to_csv(data: List[Dict[str, str]], filename: str):
    """
    Sauvegarde les données extraites dans un fichier CSV.

    Args:
        data (list): Liste des données produits.
        filename (str): Nom du fichier de sortie.
    """
    df = pd.DataFrame(data)
    df.to_csv(filename, index=False, encoding="utf-8")
    print(f"[INFO] {len(df)} produits sauvegardés dans {filename}")

if __name__ == "__main__":
    categories = load_categories("config/categories.json")
    url = categories.get("telephones_tablettes")

    if url:
        html = fetch_html(url)
        if html:
            products = parse_products(html)
            save_to_csv(products, "data/smartphones.csv")
        else:
            print("[ERREUR] Échec de récupération HTML.")
    else:
        print("[ERREUR] Catégorie non trouvée dans le fichier JSON.")


In [ ]:
import pandas as pd

# Charger les données sauvegardées
df = pd.read_csv("data/smartphones.csv")

# Afficher les premières lignes
print(df.head())


In [ ]:
df.head(10)  # Pour afficher les 10 premiers produits


In [ ]:
# Apres axtrait les données, on peut les stocker dans un fichier CSV ou une base de données pour une utilisation ultérieure.
# ce qui reste c'est le traitement avec pandas et l'analyse des données.
# On peut aussi utiliser des bibliothèques comme matplotlib ou seaborn pour visualiser les données.